# Lezione 6 · RNN, LSTM, GRU: la memoria del tempo

Il foglietto che si aggiorna a ogni passo e sbiadisce, il quaderno con le sue domande, e la pagina aperta del traduttore.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## La rete ricorrente

*Nel racconto: il foglietto di oggi, fatto dal foglietto di ieri e dalla novità di oggi.*

Il barometro: ogni ora la rete riceve di quanto è scesa la pressione, su una scala dove −1 è un crollo di cinque punti in un'ora. Il peso sul foglietto di ieri è 0,8, quello sulla novità è 1.

In [2]:
letture = [1019, 1018, 1016, 1013]
discese = [(dopo - prima) / 5 for prima, dopo in zip(letture, letture[1:])]
for v, a in zip(discese, [-0.2, -0.4, -0.6]):
    controlla("di quanto scende", v, a)

W_h, W_x, h = 0.8, 1.0, 0.0
for ora, (x, conti, tabella) in enumerate(zip(discese, [-0.197, -0.506, -0.764], [-0.20, -0.51, -0.76]), 1):
    h = math.tanh(W_h * h + W_x * x)
    controlla(f"foglietto all'ora {ora}, nei conti", h, conti, 0.001)
    controlla(f"foglietto all'ora {ora}, nella tabella del racconto", h, tabella, 0.005)

d, n = 128, 50
conta("pesi di una cella con d = 128 e ingressi di 50 numeri", d * d + d * n + d, 22912)

✓  di quanto scende: calcolato -0.2, nel libro -0.2
✓  di quanto scende: calcolato -0.4, nel libro -0.4
✓  di quanto scende: calcolato -0.6, nel libro -0.6
✓  foglietto all'ora 1, nei conti: calcolato -0.1974, nel libro -0.197
✓  foglietto all'ora 1, nella tabella del racconto: calcolato -0.1974, nel libro -0.2
✓  foglietto all'ora 2, nei conti: calcolato -0.5064, nel libro -0.506
✓  foglietto all'ora 2, nella tabella del racconto: calcolato -0.5064, nel libro -0.51
✓  foglietto all'ora 3, nei conti: calcolato -0.7637, nel libro -0.764
✓  foglietto all'ora 3, nella tabella del racconto: calcolato -0.7637, nel libro -0.76
✓  pesi di una cella con d = 128 e ingressi di 50 numeri: calcolato 22.912, nel libro 22.912


## Il ricordo che sbiadisce

*Nel racconto: «ogni giorno è un bozzello».*

In [3]:
controlla("peso 0,5, dieci passi: resta (millesimi)", 0.5 ** 10 * 1000, 1, 0.05)
controlla("peso 0,8, venti passi: resta al più", 0.8 ** 20, 0.012, 0.001)

✓  peso 0,5, dieci passi: resta (millesimi): calcolato 0.9766, nel libro 1
✓  peso 0,8, venti passi: resta al più: calcolato 0.01153, nel libro 0.012


## Il quaderno con le domande: la LSTM

*Nel racconto: che cosa cancello, che cosa scrivo, che cosa leggo.*

In [4]:
c_prima, f, i, c_proposta, o = 0.8, 0.9, 0.2, 0.5, 0.5
c = f * c_prima + i * c_proposta
controlla("il quaderno", c, 0.82)
controlla("tanh(0,82)", math.tanh(c), 0.675, 0.001)
controlla("ciò che si legge", o * math.tanh(c), 0.34)
controlla("cancello che dimentica a 0,99, dopo 100 passi resta", 0.99 ** 100, 0.37)
conta("pesi di una LSTM con d = 128 e n = 50", 4 * (d * (d + n) + d), 91648)

✓  il quaderno: calcolato 0.82, nel libro 0.82
✓  tanh(0,82): calcolato 0.6751, nel libro 0.675
✓  ciò che si legge: calcolato 0.3375, nel libro 0.34
✓  cancello che dimentica a 0,99, dopo 100 passi resta: calcolato 0.366, nel libro 0.37
✓  pesi di una LSTM con d = 128 e n = 50: calcolato 91.648, nel libro 91.648


## La versione snella: il GRU

*Nel racconto: «una versione più snella, con due domande soltanto».*

In [5]:
z, h_ieri, h_proposta = 0.1, 1.0, 0.0
controlla("con z = 0,1, di ieri resta", ((1 - z) * h_ieri + z * h_proposta) / h_ieri, 0.9)
gru = 3 * (d * (d + n) + d)
conta("pesi di un GRU con d = 128 e n = 50", gru, 68736)
controlla("in meno della LSTM (un quarto)", 1 - gru / 91648, 0.25)

✓  con z = 0,1, di ieri resta: calcolato 0.9, nel libro 0.9
✓  pesi di un GRU con d = 128 e n = 50: calcolato 68.736, nel libro 68.736
✓  in meno della LSTM (un quarto): calcolato 0.25, nel libro 0.25


## L'attenzione del 2014

*Nel racconto: le luci del porto; «il gatto nero dorme», «the black cat sleeps».*

Il decoder sta per scrivere *black*.

In [6]:
parole = ["il", "gatto", "nero", "dorme"]
gocce = softmax([0.1, 0.5, 2.0, -1.0])
for parola, v, a in zip(parole, gocce, [0.105, 0.157, 0.703, 0.035]):
    controlla(f"gocce su «{parola}»", v, a, 0.001)
foglietti = np.array([0.1, 0.6, -0.4, 0.3])
controlla("il contesto, dominato da «nero»", float(gocce @ foglietti), -0.17)

✓  gocce su «il»: calcolato 0.1051, nel libro 0.105
✓  gocce su «gatto»: calcolato 0.1569, nel libro 0.157
✓  gocce su «nero»: calcolato 0.703, nel libro 0.703
✓  gocce su «dorme»: calcolato 0.035, nel libro 0.035
✓  il contesto, dominato da «nero»: calcolato -0.1661, nel libro -0.17


## Kata per casa

**Esercizio 1.** h_t = tanh(0,5 · h_{t−1} + x_t), con h₀ = 0 e input (1; 0; 0).

In [7]:
h, storia = 0.0, []
for x in [1, 0, 0] + [0] * 7:
    h = math.tanh(0.5 * h + x)
    storia.append(h)
for t, (v, a) in enumerate(zip(storia, [0.762, 0.363, 0.180]), 1):
    controlla(f"h al passo {t}", v, a, 0.001)
controlla("dopo dieci passi resta (%)", storia[9] * 100, 0.14)

✓  h al passo 1: calcolato 0.7616, nel libro 0.762
✓  h al passo 2: calcolato 0.3634, nel libro 0.363
✓  h al passo 3: calcolato 0.1797, nel libro 0.18
✓  dopo dieci passi resta (%): calcolato 0.1399, nel libro 0.14
